In [ ]:
# Minimal imports for JSON to langextract-inspired conversion
import json
from pathlib import Path

# Import our standalone converter (no external dependencies)
from json_to_lx_converter import JSONToLXConverter, ResultPrinter, DocumentSaver

# File paths
path_data = Path('/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/annotations.conll')
output_path = Path('/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/annotations_entities.json')

print("✓ Standalone converter loaded successfully!")
print("This converter is inspired by langextract but has no external dependencies.")

: 

In [ ]:
# Load the training data and convert to JSON format
current_label = None
sort_alphabetically = False

entities = []
current_tokens = []

with path_data.open(encoding='utf-8') as handle:
    for raw_line in handle:
        line = raw_line.strip()

        if not line or raw_line.startswith('-DOCSTART-'):
            if current_label and current_tokens:
                entities.append({current_label: ' '.join(current_tokens)})
            current_tokens = []
            current_label = None
            continue

        parts = raw_line.split()
        token, tag = parts[0], parts[-1]

        if tag == 'O':
            if current_label and current_tokens:
                entities.append({current_label: ' '.join(current_tokens)})
            current_tokens = []
            current_label = None
            continue

        if '-' not in tag:
            continue

        prefix, label = tag.split('-', 1)

        if prefix == 'B':
            if current_label and current_tokens:
                entities.append({current_label: ' '.join(current_tokens)})
            current_tokens = [token]
            current_label = label
        elif prefix == 'I' and current_label == label:
            current_tokens.append(token)
        else:
            if current_label and current_tokens:
                entities.append({current_label: ' '.join(current_tokens)})
            current_tokens = []
            current_label = None

if current_label and current_tokens:
    entities.append({current_label: ' '.join(current_tokens)})

if sort_alphabetically:
    # Convert dicts to tuples for sorting
    ordered_entities = sorted(
        entities,
        key=lambda item: (list(item.keys())[0], list(item.values())[0])
    )
    entities = ordered_entities

output_path.write_text(json.dumps({'extractions': entities}, ensure_ascii=False, indent=2) + '', encoding='utf-8')
print(f'Wrote {len(entities)} entities to {output_path}')

entities[:10]


In [ ]:
# Initialize the converter and utilities
converter = JSONToLXConverter()
printer = ResultPrinter()
saver = DocumentSaver()

print("✓ All converter components initialized!")
print("Ready to convert JSON extractions to structured format with character positions.")


In [ ]:
# Example usage with sample data
sample_json = {
    "extractions": [
        {"NOMBRE": "Rodríguez, Ana Carolina"},
        {"NOMBRE": "Fernández, Diego Esteban"},
        {"FECHA": "11 de noviembre de 2023"}
    ]
}

# Sample source text
sample_source_text = """
El documento presenta información sobre Rodríguez, Ana Carolina y Fernández, Diego Esteban.
La fecha de emisión es 11 de noviembre de 2023.
"""

print("Sample JSON data:")
print(json.dumps(sample_json, indent=2, ensure_ascii=False))
print(f"\nSample source text: '{sample_source_text.strip()}'")

# Convert to structured format
annotated_doc = converter.convert(sample_json, sample_source_text, "sample_doc")

# Print detailed results
printer.print_results(annotated_doc)
printer.print_statistics(annotated_doc)


In [ ]:
# Convert the actual training data
print("Converting training data to structured format...")
print(f"Number of entities to convert: {len(entities)}")

# Create JSON format from entities
json_extractions = {'extractions': entities}

# For demonstration, let's create a simple source text
# In practice, you should use the actual text from which these entities were extracted
demo_source_text = """
This is a sample document containing various entities that were extracted.
The entities include names, dates, and other information that was tagged.
"""

# Convert the entities to structured format
annotated_doc = converter.convert(json_extractions, demo_source_text, "training_doc")

# Show results
printer.print_results(annotated_doc, max_show=3)
printer.print_statistics(annotated_doc)

# You can also access the extractions directly
print("\nDirect access to extractions:")
for extraction in annotated_doc.extractions[:3]:  # Show first 3
    print(f"- {extraction.extraction_class}: '{extraction.extraction_text}'")
    if extraction.char_interval:
        print(f"  Position: {extraction.char_interval.start_pos}-{extraction.char_interval.end_pos}")
    print(f"  Alignment: {extraction.alignment_status}")
    print()


In [ ]:
# Save and load functionality using our converter utilities

# Example: Save the converted document
output_file = Path("converted_annotations.json")
saver.save_to_json(annotated_doc, output_file)

# Show final statistics
printer.print_statistics(annotated_doc)
print(f"\nOutput saved to: {output_file}")

# Example: Load the document back
print("\nLoading the document back to verify...")
loaded_doc = saver.load_from_json(output_file)
print(f"Loaded document with {len(loaded_doc.extractions)} extractions")
print(f"Document ID: {loaded_doc.document_id}")

# Example: Batch processing
print("\n=== Batch Processing Example ===")
json_batch = [
    {"extractions": [{"PERSONA": "Juan Pérez"}]},
    {"extractions": [{"FECHA": "2023-01-01"}]},
]

text_batch = [
    "El nombre es Juan Pérez",
    "La fecha es 2023-01-01",
]

batch_results = converter.convert_batch(json_batch, text_batch, ["doc1", "doc2"])
for i, result in enumerate(batch_results):
    print(f"Batch document {i+1}: {len(result.extractions)} extractions")
    printer.print_results(result, max_show=1)
